Document
↓
Chunking
↓
Embedding
↓
Vector DB
↓
Retrieval
↓
LLM

# Embedding Model R&D

Goal:
Compare embedding models for insurance document retrieval.

Models:
- all-MiniLM-L6-v2
- BAAI/bge-small-en-v1.5
- intfloat/e5-small-v2

In [1]:
! pip install sentence-transformers


[notice] A new release of pip is available: 24.3.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd
import time

c:\Users\Vikram\insurance_rag\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
#test Query
query = "What are the exclusions in this policy?"

chunks = [
    "Pre-existing diseases are excluded unless declared.",
    "Premium can be paid monthly or yearly.",
    "Hospitalization expenses are covered.",
    "Cosmetic surgery is not covered under the policy."
]

In [4]:
models = {
    "MiniLM": "sentence-transformers/all-MiniLM-L6-v2",
    "BGE": "BAAI/bge-small-en-v1.5",
    "E5": "intfloat/e5-small-v2"
}

In [5]:
results = []

for name, model_name in models.items():

    start = time.time()

    model = SentenceTransformer(model_name)

    query_emb = model.encode([query])

    chunk_embs = model.encode(chunks)

    scores = cosine_similarity(query_emb, chunk_embs)[0]

    best_idx = scores.argmax()

    end = time.time()

    results.append({
        "Model": name,
        "Best Match": chunks[best_idx],
        "Score": round(float(scores[best_idx]), 4),
        "Load+Encode Time": round(end - start, 2)
    })

pd.DataFrame(results)

c:\Users\Vikram\insurance_rag\venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Vikram\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9612.47it/s]
c:\Users\Vikram\insurance

,Model,Best Match,Score,Load+Encode Time
0,MiniLM,Cosmetic surgery is not covered under the policy.,0.4627,29.47
1,BGE,Pre-existing diseases are excluded unless decl...,0.7045,30.64
2,E5,Cosmetic surgery is not covered under the policy.,0.8388,30.39


In [6]:
test_cases = [
    {
        "query": "What are the exclusions in this policy?",
        "expected_keywords": ["excluded", "not covered", "pre-existing", "cosmetic"]
    },
    {
        "query": "What expenses are covered during hospitalization?",
        "expected_keywords": ["hospitalization", "room", "nursing", "medical"]
    },
    {
        "query": "Is premium payment monthly or yearly?",
        "expected_keywords": ["premium", "monthly", "yearly"]
    },
    {
        "query": "Does this policy cover cosmetic surgery?",
        "expected_keywords": ["cosmetic", "not covered"]
    }
]

chunks = [
    "Pre-existing diseases are excluded unless declared.",
    "Premium can be paid monthly or yearly.",
    "Hospitalization expenses including room rent, nursing and medical practitioner fees are covered.",
    "Cosmetic surgery is not covered under the policy.",
    "The policy provides ambulance cover and day care treatment benefits."
]

In [7]:
all_results = []

for model_label, model_name in models.items():
    model = SentenceTransformer(model_name)
    chunk_embs = model.encode(chunks)

    for case in test_cases:
        query_emb = model.encode([case["query"]])
        scores = cosine_similarity(query_emb, chunk_embs)[0]
        best_idx = scores.argmax()
        best_chunk = chunks[best_idx]

        keyword_hit = any(
            keyword.lower() in best_chunk.lower()
            for keyword in case["expected_keywords"]
        )

        all_results.append({
            "Model": model_label,
            "Query": case["query"],
            "Best Match": best_chunk,
            "Score": round(float(scores[best_idx]), 4),
            "Keyword Hit": keyword_hit
        })

pd.DataFrame(all_results)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8016.39it/s]


,Model,Query,Best Match,Score,Keyword Hit
0,MiniLM,What are the exclusions in this policy?,Cosmetic surgery is not covered under the policy.,0.4627,True
1,MiniLM,What expenses are covered during hospitalization?,"Hospitalization expenses including room rent, ...",0.8009,True
2,MiniLM,Is premium payment monthly or yearly?,Premium can be paid monthly or yearly.,0.9064,True
3,MiniLM,Does this policy cover cosmetic surgery?,Cosmetic surgery is not covered under the policy.,0.8054,True
4,BGE,What are the exclusions in this policy?,Pre-existing diseases are excluded unless decl...,0.7045,True
5,BGE,What expenses are covered during hospitalization?,"Hospitalization expenses including room rent, ...",0.8809,True
6,BGE,Is premium payment monthly or yearly?,Premium can be paid monthly or yearly.,0.8933,True
7,BGE,Does this policy cover cosmetic surgery?,Cosmetic surgery is not covered under the policy.,0.8445,True
8,E5,What are the exclusions in this policy?,Cosmetic surgery is not covered under the policy.,0.8388,True
9,E5,What expenses are covered during hospitalization?,"Hospitalization expenses including room rent, ...",0.9369,True


| Model  | 4/4 Correct? | Avg Score (roughly) |
| ------ | ------------ | ------------------- |
| MiniLM | ✅            | ~0.74               |
| BGE    | ✅            | ~0.83               |
| E5     | ✅            | ~0.91               |


# Embedding R&D Conclusion

Models Evaluated:
- all-MiniLM-L6-v2
- BAAI/bge-small-en-v1.5
- intfloat/e5-small-v2

Evaluation Method:
Insurance-related queries were matched against candidate policy chunks using cosine similarity.

Results:
- All models successfully retrieved relevant chunks.
- MiniLM achieved the lowest similarity scores.
- BGE demonstrated improved semantic retrieval.
- E5 achieved the highest similarity scores across all test cases.

Decision:
E5-small-v2 demonstrated the strongest retrieval performance in the current evaluation.

Recommendation:
For production-scale insurance retrieval systems, E5-small-v2 should be considered as a strong candidate embedding model.



Model	Query	Best Match	Score	Keyword Hit
0	MiniLM	What are the exclusions in this policy?	Cosmetic surgery is not covered under the policy.	0.4627	True
1	MiniLM	What expenses are covered during hospitalization?	Hospitalization expenses including room rent, ...	0.8009	True
2	MiniLM	Is premium payment monthly or yearly?	Premium can be paid monthly or yearly.	0.9064	True
3	MiniLM	Does this policy cover cosmetic surgery?	Cosmetic surgery is not covered under the policy.	0.8054	True
4	BGE	What are the exclusions in this policy?	Pre-existing diseases are excluded unless decl...	0.7045	True
5	BGE	What expenses are covered during hospitalization?	Hospitalization expenses including room rent, ...	0.8809	True
6	BGE	Is premium payment monthly or yearly?	Premium can be paid monthly or yearly.	0.8933	True
7	BGE	Does this policy cover cosmetic surgery?	Cosmetic surgery is not covered under the policy.	0.8445	True
8	E5	What are the exclusions in this policy?	Cosmetic surgery is not covered under the policy.	0.8388	True
9	E5	What expenses are covered during hospitalization?	Hospitalization expenses including room rent, ...	0.9369	True
10	E5	Is premium payment monthly or yearly?	Premium can be paid monthly or yearly.	0.9436	True
11	E5	Does this policy cover cosmetic surgery?	Cosmetic surgery is not covered under the policy.	0.9171	True
